# 07 · TQ3: practical challenges, quantified

**What it does.** Pulls together the measurements behind the four TQ3 themes of the brief (circuit depth,
parameter tuning, encoding, simulator limits) from the stored results and the circuits themselves.

**Why it was implemented.** TQ3 asks which practical challenges arise even for small instances. Each claim in
the report (`docs/report_vrp.md` §6) and the running log (`docs/challenges.md`) is backed by a number here.

**Resources.**
- Data: `results/sweep/summary.csv`; the first ("baseline") pipeline run `results/vrp_stage2_v0_qaoa_20261018T0935*.json`, made before cost normalisation was added; the reported run `results/vrp_stage2_v0_qaoa_20261018T0956*.json`.
- Code: `common/solvers.py` (normalisation, depth report), `domains/vrp/problem.py`.
- Course: simulators and their ~30–40-qubit limit, `Lecture 6` slide 8.

In [ ]:
# Run from anywhere: move to the repository root so `common/`, `domains/`, `data/` resolve.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
import csv
import json

rows = list(csv.DictReader(Path("results/sweep/summary.csv").open(encoding="utf-8")))
def best_by_reps(qubo):
    out = {}
    for r in rows:
        if r["qubo"] == qubo and "reps" in r["experiments"].split("+"):
            p = int(r["reps"])
            if p not in out or float(r["p_optimal"]) > out[p][0]:
                out[p] = (float(r["p_optimal"]), int(r["hw_cx"]), r["optimizer"])
    return dict(sorted(out.items()))

## 1. Circuit depth
Each layer applies one two-qubit rotation per QUBO coupling, i.e. 2 CNOTs on a generic device: Stage 1 has
16 couplings (32 CNOTs per layer), the 9-qubit route 30 (60 per layer). On the noiseless simulator depth only
costs runtime; on hardware every CNOT adds error.

In [ ]:
depth = {q: best_by_reps(q) for q in ("stage1", "stage2")}
for q, d in depth.items():
    print(q)
    for p, (p_opt, cx, opt) in d.items():
        print(f"  p = {p}: CNOTs {cx:>3} | best p_optimal {p_opt:.3f} ({opt})")

## 2. Parameter tuning: the energy scale
Penalties make the Ising coefficients large (up to 134 for Stage 1, 118 for the route). Without normalisation,
γ must be about 0.003 while COBYLA's first step is 1.0, so the optimiser jumps out of the useful range. The
first pipeline run (p = 2, COBYLA, no normalisation) is compared with the same settings after normalisation.

In [ ]:
baseline_path = sorted(Path("results").glob("vrp_stage2_v0_qaoa_*T0935*.json"))[0]
baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
after = next(r for r in rows if r["qubo"] == "stage2" and r["reps"] == "2" and r["optimizer"] == "COBYLA"
             and r["maxiter"] == "300" and r["initial_gamma"] == "1.0" and r["penalty"] == "102")
g = baseline["formulation_check"]["ground_energy"]
optimal_shots = sum(x["count"] for x in baseline["distribution"] if abs(x["energy"] - g) < 1e-6)
print(f"before normalisation: initial gamma {baseline['initial_gammas']}, normalised: {'cost_scale' in baseline}, "
      f"optimal shots {optimal_shots} of {baseline['shots']} -> p_optimal {baseline['stats']['p_optimal']:.4f}")
print(f"after normalisation:  p_optimal {float(after['p_optimal']):.3f}")
print(f"uniform sampling:     p_optimal {2 / 512:.4f} (2 optimal tours of 512 bitstrings)")

## 2b. Parameter tuning: starting point and optimiser

In [ ]:
for r in sorted((r for r in rows if r["qubo"] == "stage2" and "initial_gamma" in r["experiments"].split("+")),
                key=lambda r: float(r["initial_gamma"])):
    print(f"route, initial gamma {r['initial_gamma']:>4}: p_optimal {float(r['p_optimal']):.3f}")
nm = {int(r["reps"]): float(r["p_optimal"]) for r in rows
      if r["qubo"] == "stage2" and "reps" in r["experiments"].split("+") and r["optimizer"] == "Nelder-Mead"}
print("route, Nelder-Mead p_optimal by p:", nm)

## 3. Encoding
- One-hot routes: 6 valid tours of 512 bitstrings for 3 customers (98.8% infeasible).
- The slack-free capacity encoding kept Stage 1 at 8 qubits, but only works for 2 vehicles with $C = \lceil n/2 \rceil$; `problem.py` refuses other cases, shown below.
- Symmetry: fixing customer 1 removed 2 qubits; tour reversal was kept (2 optimal tours per route).

In [ ]:
from dataclasses import replace
from domains.vrp.problem import build_bqm, load_instance

inst = load_instance("data/vrp/P-n6_1.vrp")
try:
    build_bqm(replace(inst, capacity=4), {"one_vehicle": 162, "capacity": 162})
    refused = False
except ValueError as e:
    refused = True
    print("capacity 4 refused:", e)

## 4. Simulator limits
A statevector simulator stores $2^n$ complex amplitudes (16 bytes each). Memory doubles with every qubit,
which is why the VRP was decomposed instead of solved as one ~30-qubit QUBO.

In [ ]:
def human(nbytes):
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if nbytes < 1024:
            return f"{nbytes:g} {unit}"
        nbytes /= 1024
    return f"{nbytes:g} PiB"

for n, what in [(4, "route {3,4}"), (8, "Stage 1"), (9, "route {1,2,5}"), (12, "project limit"),
                (30, "monolithic VRP (5 customers x 2 vans x 3 positions)"), (40, "course: practical ceiling")]:
    print(f"{n:>3} qubits: {human(16 * 2 ** n):>9}  {what}")

## Expected results
- CNOTs per layer: 32 (Stage 1) and 60 (route); best route run at p = 4 needs 240 CNOTs.
- Before normalisation: 2 of 4096 shots optimal (p_optimal ≈ 0.0005), *below* uniform sampling (0.0039); after: **0.079**.
- Initial γ: route p_optimal 0.029 (γ = 0.4) to 0.145 (γ = 2). Nelder-Mead near uniform (≈ 0.004) at p ≥ 4.
- Capacity 4 is refused by the slack-free encoding.
- 9 qubits = 8 KiB; 30 qubits = 16 GiB; 40 qubits = 16 TiB.

Not done (optional extensions noted in the report): multi-start QAOA, an annealing-schedule sweep, and solving a
monolithic VRP QUBO with simulated annealing only.

In [ ]:
assert {p: cx for p, (_, cx, _) in depth["stage2"].items()} == {p: 60 * p for p in range(1, 6)}
assert optimal_shots == 2 and "cost_scale" not in baseline and abs(float(after["p_optimal"]) - 0.079) < 0.002
assert refused
assert 16 * 2 ** 30 == 16 * 1024 ** 3